# Aula 5 — Redes neuronais: backward propagation

Testes do `backward_propagation` da `DenseLayer` (slide 12) e das camadas de ativação (slide 20).


## DenseLayer.backward_propagation (slides 11–13 ppt da aula6-P-solved.pdf)

Gradient check: compara o dE/dW calculado pelo `backward_propagation` com uma derivada numérica. Se a fórmula estiver certa, a diferença é praticamente zero.



In [1]:
import numpy as np
from si.neural_networks.layers import DenseLayer

# optimizer "falso": não mexe nos pesos, só guarda o último gradiente que recebeu
# (o SGD e o Adam só chegam na próxima aula)
class GuardaGradiente:
    def update(self, w, grad):
        self.grad = grad
        return w

# MSE em numpy (as funções de loss também só chegam na próxima aula)
mse = lambda Y, P: np.mean((Y - P) ** 2)
# dE/dY: divide por Y.size porque a MSE é uma média de todos os elementos
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size



In [2]:
np.random.seed(0)
X, Y = np.random.rand(4, 3), np.random.rand(4, 2)  # 4 amostras, 3 features, 2 outputs

layer = DenseLayer(2, (3,)).initialize(GuardaGradiente())
out = layer.forward_propagation(X, training=True)
layer.backward_propagation(mse_derivative(Y, out))
analytic = layer.w_opt.grad  # dE/dW calculado pelo backward_propagation
# (lê-se de layer.w_opt e não do objeto original: o deepcopy do initialize criou uma cópia)

# derivada numérica: mexe cada peso um bocadinho para cima e para baixo e vê quanto a loss muda
numeric, eps = np.zeros_like(layer.weights), 1e-6
for i in range(3):
    for j in range(2):
        layer.weights[i, j] += eps
        lp = mse(Y, layer.forward_propagation(X, True))  # loss com w + eps
        layer.weights[i, j] -= 2 * eps
        lm = mse(Y, layer.forward_propagation(X, True))  # loss com w - eps
        layer.weights[i, j] += eps                         # volta ao valor original
        numeric[i, j] = (lp - lm) / (2 * eps)              # declive entre os dois pontos

print("dE/dW (backward):\n", analytic.round(6))
print("dE/dW (numérico):\n", numeric.round(6))
print("diferença máxima:", np.abs(numeric - analytic).max())  # ~1e-11

dE/dW (backward):
 [[-0.189787 -0.105199]
 [-0.319588 -0.216488]
 [-0.257312 -0.162776]]
dE/dW (numérico):
 [[-0.189787 -0.105199]
 [-0.319588 -0.216488]
 [-0.257312 -0.162776]]
diferença máxima: 1.513400516017782e-11


**Conclusão (DenseLayer.backward_propagation):** o dE/dW calculado pelo `backward_propagation` é igual à derivada numérica (diferença da ordem de 1e-11), por isso a fórmula dE/dW = Xᵀ·dE/dY está bem implementada.

- O `backward_propagation` calcula 3 gradientes, cada um com a shape daquilo a que se refere: dE/dX = dE/dY·Wᵀ (shape de X, passa para a camada anterior), dE/dW = Xᵀ·dE/dY (shape de W) e dE/dB = soma de dE/dY nas amostras (shape de B).
- O dE/dX tem de ser calculado antes de atualizar os pesos, senão usava os pesos novos.
- O teste usa um optimizer "falso" que não mexe nos pesos, porque o SGD, o Adam e as funções de loss só chegam na próxima aula.

## Camadas de ativação (slides 14–20)

Implementadas em `src/si/neural_networks/activation.py`: classe base abstrata `ActivationLayer`, `ReLUActivation` e `SigmoidActivation`. O teste compara a derivada de cada ativação com a derivada numérica.

In [3]:
from si.neural_networks.activation import ReLUActivation, SigmoidActivation

# valores de teste: evitam o 0, onde a derivada da ReLU não existe
x, eps = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7]), 1e-6

for act in [SigmoidActivation(), ReLUActivation()]:
    # derivada numérica: declive entre f(x - eps) e f(x + eps)
    num = (act.activation_function(x + eps) - act.activation_function(x - eps)) / (2 * eps)
    print(act.layer_name())
    print("  f(x):  ", act.activation_function(x).round(4))
    print("  f'(x): ", act.derivative(x).round(4))
    print("  diferença máxima:", np.abs(num - act.derivative(x)).max())  # ~1e-10

SigmoidActivation
  f(x):   [0.0759 0.2315 0.4256 0.5987 0.7503 0.937 ]
  f'(x):  [0.0701 0.1779 0.2445 0.2403 0.1874 0.059 ]
  diferença máxima: 6.56039597646263e-11
ReLUActivation
  f(x):   [0.  0.  0.  0.4 1.1 2.7]
  f'(x):  [0 0 0 1 1 1]
  diferença máxima: 1.397779669787269e-10


**Conclusão (ativações):** as derivadas da `SigmoidActivation` e da `ReLUActivation` coincidem com as derivadas numéricas (diferença da ordem de 1e-10).

- A ReLU põe os negativos a zero e deixa os positivos iguais; a derivada é 0 ou 1, por isso não encolhe o erro nos neurónios ativos.
- A sigmoide comprime os valores para ]0, 1[; a derivada f(x)·(1 − f(x)) é no máximo 0.25, por isso, com muitas camadas, o erro vai ficando cada vez mais pequeno (vanishing gradient).
- No backward, a multiplicação é elemento a elemento (`*`), porque cada output de uma ativação depende só do seu próprio input.
